In [0]:
# Ingests every entity from `bronze.raw_files` into its own Bronze Delta table via
# Auto Loader. Safe to rerun since already-ingested files are skipped per entity via
# each entity's own checkpoint.

from pyspark.sql import functions as F

CATALOG = "ecommerce_project"
BASE_VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/raw_files"

ENTITIES = ["customers", "products", "orders", "order_items", "payments", "events"]

def ingest_entity(entity: str) -> None:
    source_path = f"{BASE_VOLUME_PATH}/{entity}"
    checkpoint_path = f"{BASE_VOLUME_PATH}/_checkpoints/{entity}"
    schema_path = f"{BASE_VOLUME_PATH}/_schemas/{entity}"
    target_table = f"{CATALOG}.bronze.{entity}"

    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", schema_path)
        .load(source_path)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    query = (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .table(target_table)
    )
    query.awaitTermination()

    row_count = spark.table(target_table).count()
    print(f"{entity}: ingested into {target_table}, {row_count} rows total")

for entity in ENTITIES:
    ingest_entity(entity)